<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Ejercicios/Soluciones/06_Taller_Word2Vec_GloVe_Seq2Seq_Solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Taller práctico: Word2Vec, GloVe y Seq2Seq (solución)</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Instrucciones</h2>

Solución de referencia del taller [`06_Taller_Word2Vec_GloVe_Seq2Seq.ipynb`](../06_Taller_Word2Vec_GloVe_Seq2Seq.ipynb).

Hay muchas soluciones válidas: esta es una de ellas. Compara tus resultados y, sobre todo, tus conclusiones.

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Ejercicio 1. Hiperparámetros de Word2Vec</span>](#ejercicio-1)
- [<span style="color: #005C99;">Ejercicio 2. Analogías: 3CosAdd frente a 3CosMul</span>](#ejercicio-2)
- [<span style="color: #005C99;">Ejercicio 3. Midiendo el sesgo con WEAT</span>](#ejercicio-3)
- [<span style="color: #005C99;">Ejercicio 4. Seq2Seq a nivel de caracteres: normalizar fechas</span>](#ejercicio-4)
- [<span style="color: #005C99;">Ejercicio 5. Generalización a formatos no vistos</span>](#ejercicio-5)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [1]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib tensorflow gensim

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # oculta mensajes informativos de TensorFlow

import re
import string
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)  # fija las semillas de Python, NumPy y TensorFlow

import datetime
import random

import gensim.downloader as api
from gensim.models import Word2Vec
from gensim.models.word2vec import Text8Corpus
from gensim.test.utils import datapath

ruta_text8 = api.load("text8", return_path=True)   # ~17 millones de palabras de Wikipedia en inglés
glove = api.load("glove-wiki-gigaword-100")
print(f"GloVe: {len(glove):,} palabras")

GloVe: 400,000 palabras


---

<a id="ejercicio-1"></a>
<h2 style="color: #007ACC;">Ejercicio 1. Hiperparámetros de Word2Vec</h2>

**Objetivo:** medir cómo cambian los embeddings con la arquitectura (CBOW o skip-gram) y el tamaño de la ventana.

1. Entrena cuatro modelos Word2Vec sobre text8 con 100 dimensiones, `min_count=5`, 3 épocas y `negative=5`, combinando:
   - `sg=0` (CBOW) y `sg=1` (skip-gram),
   - `window=2` y `window=10`.
2. Para cada modelo reporta el tiempo de entrenamiento y:
   - la exactitud en las analogías de Google (`datapath("questions-words.txt")`, con `restrict_vocab=30_000`), **separando** las secciones semánticas (las 5 primeras: capitales, monedas, ciudades, familia) de las sintácticas (las que empiezan por `gram`);
   - la correlación de Spearman en `datapath("wordsim353.tsv")`.
   Pista: `evaluate_word_analogies` devuelve también una lista de secciones con sus aciertos (`correct`) y errores (`incorrect`).
3. Muestra los 6 vecinos de `"computer"` y de `"run"` con ventana 2 y con ventana 10 (skip-gram).
4. ¿Qué configuración favorece las analogías semánticas y cuál las sintácticas? Relaciónalo con lo visto en el taller 02 sobre el tamaño de la ventana.

In [3]:
def exactitud_secciones(kv):
    _, secciones = kv.evaluate_word_analogies(datapath("questions-words.txt"), restrict_vocab=30_000)
    sem = [s for s in secciones if not s["section"].startswith("gram") and s["section"] != "Total accuracy"]
    sin = [s for s in secciones if s["section"].startswith("gram")]
    acc = lambda ss: sum(len(s["correct"]) for s in ss) / sum(len(s["correct"]) + len(s["incorrect"]) for s in ss)
    return acc(sem), acc(sin)


modelos_w2v, filas = {}, []
for sg in [0, 1]:
    for ventana in [2, 10]:
        inicio = time.perf_counter()
        m = Word2Vec(Text8Corpus(ruta_text8), vector_size=100, window=ventana, min_count=5, sg=sg,
                     negative=5, epochs=3, workers=8, seed=SEED)
        nombre = f"{'skip-gram' if sg else 'CBOW'}, ventana={ventana}"
        modelos_w2v[nombre] = m
        sem, sin = exactitud_secciones(m.wv)
        filas.append({"modelo": nombre, "tiempo (s)": round(time.perf_counter() - inicio),
                      "analogías semánticas": sem, "analogías sintácticas": sin,
                      "WordSim-353": m.wv.evaluate_word_pairs(datapath("wordsim353.tsv"))[1].statistic})
pd.DataFrame(filas).set_index("modelo").round(3)

,tiempo (s),analogías semánticas,analogías sintácticas,WordSim-353
modelo,,,,
"CBOW, ventana=2",12,0.103,0.238,0.493
"CBOW, ventana=10",12,0.165,0.299,0.623
"skip-gram, ventana=2",14,0.145,0.304,0.579
"skip-gram, ventana=10",29,0.290,0.374,0.682


In [4]:
for palabra in ["computer", "run"]:
    for ventana in [2, 10]:
        vecinos = [w for w, _ in modelos_w2v[f"skip-gram, ventana={ventana}"].wv.most_similar(palabra, topn=6)]
        print(f"{palabra:<9} ventana={ventana:<3} -> {', '.join(vecinos)}")

computer  ventana=2   -> computers, programmer, hardware, console, computing, developer
computer  ventana=10  -> computers, hardware, bootstrap, supercomputer, microcontroller, computing
run       ventana=2   -> running, lottery, subscribers, syndication, tickets, mediocre
run       ventana=10  -> running, runs, ran, userland, marketshare, stalls


**Interpretación:**

- **Skip-gram** supera a **CBOW** en todas las pruebas, a cambio de un entrenamiento más lento.
- En este experimento la **ventana grande** gana en todo, incluidas las analogías **sintácticas**. Esto contradice la idea habitual de que la ventana pequeña favorece las relaciones sintácticas. Una explicación probable: con solo 3 épocas sobre 17 millones de palabras, la ventana grande genera muchos más pares de entrenamiento por época, así que el modelo está mejor entrenado. Para aislar el efecto de la ventana habría que igualar el número de actualizaciones.
- Los vecinos de *computer* son razonables en ambos casos. Los de *run* muestran el problema de la **polisemia**: con ventana 2 aparecen *lottery*, *tickets* y *subscribers* (de usos como *run a lottery*); con ventana 10, las formas flexionadas *runs* y *ran*. Un único vector por palabra mezcla todos sus sentidos.
- Lección: las "reglas generales" sobre hiperparámetros dependen del corpus y del presupuesto de entrenamiento, así que conviene medirlas en cada caso, como vimos en el cuaderno 12 con las evaluaciones intrínsecas que se contradecían.

---

<a id="ejercicio-2"></a>
<h2 style="color: #007ACC;">Ejercicio 2. Analogías: 3CosAdd frente a 3CosMul</h2>

**Objetivo:** implementar desde cero dos métodos de resolución de analogías y compararlos sobre GloVe.

Para una analogía *a : a\* :: b : ?* (por ejemplo *man : king :: woman : ?*), con vectores normalizados:

- **3CosAdd** (Mikolov et al., 2013): $\arg\max_{x} \cos(x, a^*) - \cos(x, a) + \cos(x, b)$
- **3CosMul** (Levy & Goldberg, 2014): $\arg\max_{x} \dfrac{\cos'(x, a^*)\, \cos'(x, b)}{\cos'(x, a) + \varepsilon}$, con $\cos' = (\cos + 1)/2$ para que sea positivo y $\varepsilon = 0.001$.

En ambos casos se excluyen $a$, $a^*$ y $b$ de los candidatos.

1. Construye la matriz normalizada `M` con las 50 000 palabras más frecuentes de GloVe (`glove.index_to_key[:50_000]`).
2. Lee las analogías de `questions-words.txt` (las líneas que empiezan con `:` marcan secciones) y quédate con las que tienen sus cuatro palabras en ese vocabulario. Toma una muestra aleatoria de 3 000.
3. Implementa `resolver(A, A_est, B, metodo)` **vectorizado**: recibe arreglos de índices y devuelve los índices predichos para todo el lote.
4. Compara la exactitud de ambos métodos (global y por sección).
5. Muestra 5 analogías donde 3CosMul acierta y 3CosAdd falla. ¿Qué tienen en común?

In [5]:
palabras_g = glove.index_to_key[:50_000]
idx_g = {w: i for i, w in enumerate(palabras_g)}
M = glove.vectors[:50_000] / np.linalg.norm(glove.vectors[:50_000], axis=1, keepdims=True)

preguntas, seccion = [], None
with open(datapath("questions-words.txt")) as f:
    for linea in f:
        if linea.startswith(":"):
            seccion = linea[1:].strip()
            continue
        ws = linea.lower().split()
        if all(w in idx_g for w in ws):
            preguntas.append((seccion, *[idx_g[w] for w in ws]))
random.Random(SEED).shuffle(preguntas)
preguntas = preguntas[:3000]
secciones = np.array([p[0] for p in preguntas])
A, A_est, B, D = (np.array([p[i] for p in preguntas]) for i in range(1, 5))
print(f"Analogías evaluadas: {len(preguntas)}")


def resolver(A, A_est, B, metodo="add", lote=500, eps=1e-3):
    predicciones = []
    for i in range(0, len(A), lote):
        a, a_est, b = A[i:i + lote], A_est[i:i + lote], B[i:i + lote]
        cos_a, cos_ae, cos_b = M[a] @ M.T, M[a_est] @ M.T, M[b] @ M.T
        if metodo == "add":
            puntaje = cos_ae - cos_a + cos_b
        else:
            f = lambda c: (c + 1) / 2
            puntaje = f(cos_ae) * f(cos_b) / (f(cos_a) + eps)
        filas = np.arange(len(a))
        for excluir in (a, a_est, b):
            puntaje[filas, excluir] = -np.inf
        predicciones.append(puntaje.argmax(axis=1))
    return np.concatenate(predicciones)


pred_add, pred_mul = resolver(A, A_est, B, "add"), resolver(A, A_est, B, "mul")
tabla = pd.DataFrame({"sección": secciones, "3CosAdd": pred_add == D, "3CosMul": pred_mul == D})
resumen = tabla.groupby("sección")[["3CosAdd", "3CosMul"]].mean()
resumen.loc["TOTAL"] = tabla[["3CosAdd", "3CosMul"]].mean()
resumen.round(3)

Analogías evaluadas: 3000


,3CosAdd,3CosMul
sección,,
capital-common-countries,0.938,0.938
capital-world,0.893,0.857
city-in-state,0.307,0.257
currency,0.217,0.261
family,0.838,0.838
gram1-adjective-to-adverb,0.240,0.247
gram2-opposite,0.233,0.183
gram3-comparative,0.799,0.771
gram4-superlative,0.562,0.617


In [6]:
solo_mul = np.where((pred_mul == D) & (pred_add != D))[0][:5]
for i in solo_mul:
    w = lambda j: palabras_g[j]
    print(f"{w(A[i])} : {w(A_est[i])} :: {w(B[i])} : ?   correcta={w(D[i])} | 3CosAdd={w(pred_add[i])} | 3CosMul={w(pred_mul[i])}")

aware : unaware :: known : ?   correcta=unknown | 3CosAdd=name | 3CosMul=unknown
elephant : elephants :: melon : ?   correcta=melons | 3CosAdd=watermelon | 3CosMul=melons
building : buildings :: melon : ?   correcta=melons | 3CosAdd=watermelon | 3CosMul=melons
sing : singing :: increase : ?   correcta=increasing | 3CosAdd=increased | 3CosMul=increasing
knowing : knew :: screaming : ?   correcta=screamed | 3CosAdd=yelling | 3CosMul=screamed


**Interpretación:** sobre GloVe, **3CosAdd** obtiene una exactitud total ligeramente mayor. 3CosMul gana en algunas secciones (superlativos, monedas, plurales de verbos) y pierde en otras. Levy & Goldberg reportaron mejoras más claras con 3CosMul sobre representaciones dispersas y sobre Word2Vec. El resultado depende de los embeddings.

Los casos en que 3CosMul acierta y 3CosAdd falla siguen un patrón claro: el error de 3CosAdd es una palabra **muy similar a *b*** (*melon* → *watermelon*; *increase* → *increased*), que gana por un solo término muy grande de la suma. La forma multiplicativa exige que el candidato sea similar a **las dos** palabras positivas, lo que equilibra los términos. A cambio, a veces penaliza en exceso candidatos correctos que también son cercanos a *a*.

---

<a id="ejercicio-3"></a>
<h2 style="color: #007ACC;">Ejercicio 3. Midiendo el sesgo con WEAT</h2>

**Objetivo:** cuantificar una asociación estereotipada en GloVe con la prueba WEAT y evaluar su significancia con un test de permutaciones.

La prueba **WEAT** (*Word Embedding Association Test*, Caliskan, Bryson & Narayanan, 2017) adapta un test psicológico de asociaciones implícitas. Dados dos conjuntos de palabras **objetivo** $X$, $Y$ y dos conjuntos de **atributos** $A$, $B$:

$$s(w, A, B) = \overline{\cos(w, a)}_{a \in A} - \overline{\cos(w, b)}_{b \in B}$$

$$d = \frac{\overline{s(x, A, B)}_{x \in X} - \overline{s(y, A, B)}_{y \in Y}}{\text{desv. est.}_{w \in X \cup Y}\; s(w, A, B)}$$

El tamaño del efecto $d$ mide si las palabras de $X$ se asocian más con $A$ (y las de $Y$ con $B$) que al revés.

Usa los conjuntos del estudio original:

```python
X = ["executive", "management", "professional", "corporation", "salary", "office", "business", "career"]  # carrera
Y = ["home", "parents", "children", "family", "cousins", "marriage", "wedding", "relatives"]             # familia
A = ["john", "paul", "mike", "kevin", "steve", "greg", "jeff", "bill"]                                   # nombres masculinos
B = ["amy", "joan", "lisa", "sarah", "diana", "kate", "ann", "donna"]                                     # nombres femeninos
```

1. Implementa `weat(X, Y, A, B, kv)` que devuelva el tamaño del efecto $d$.
2. Implementa un **test de permutaciones**: repite 10 000 veces el cálculo de $\overline{s(x)} - \overline{s(y)}$ redistribuyendo al azar las palabras de $X \cup Y$ en dos grupos del mismo tamaño, y calcula el valor $p$ como la fracción de permutaciones con una diferencia mayor o igual a la observada.
3. Calcula $d$ y $p$ en GloVe y en el Word2Vec skip-gram (ventana 10) del ejercicio 1.
4. Como **control**, repite la prueba con dos conjuntos objetivo sin relación esperada con el género, por ejemplo instrumentos musicales frente a herramientas. ¿Qué esperarías encontrar?

In [7]:
X = ["executive", "management", "professional", "corporation", "salary", "office", "business", "career"]
Y = ["home", "parents", "children", "family", "cousins", "marriage", "wedding", "relatives"]
A = ["john", "paul", "mike", "kevin", "steve", "greg", "jeff", "bill"]
B = ["amy", "joan", "lisa", "sarah", "diana", "kate", "ann", "donna"]


def asociaciones(palabras, A, B, kv):
    unit = lambda ws: np.array([kv[w] / np.linalg.norm(kv[w]) for w in ws])
    W, VA, VB = unit(palabras), unit(A), unit(B)
    return (W @ VA.T).mean(axis=1) - (W @ VB.T).mean(axis=1)


def weat(X, Y, A, B, kv, n_perm=10_000, rng=None):
    rng = rng or np.random.default_rng(SEED)
    s = asociaciones(X + Y, A, B, kv)
    sx, sy = s[:len(X)], s[len(X):]
    d = (sx.mean() - sy.mean()) / s.std(ddof=1)
    observado = sx.mean() - sy.mean()
    mayores = 0
    for _ in range(n_perm):
        perm = rng.permutation(len(s))
        mayores += s[perm[:len(X)]].mean() - s[perm[len(X):]].mean() >= observado
    return d, mayores / n_perm


instrumentos = ["guitar", "piano", "violin", "flute", "drum", "trumpet", "cello", "harp"]
herramientas = ["hammer", "saw", "wrench", "screwdriver", "drill", "chisel", "pliers", "axe"]
w2v_kv = modelos_w2v["skip-gram, ventana=10"].wv

filas = []
for nombre_emb, kv in [("GloVe", glove), ("Word2Vec text8", w2v_kv)]:
    for prueba, (x, y) in {"carrera vs. familia": (X, Y), "control: instrumentos vs. herramientas": (instrumentos, herramientas)}.items():
        if all(w in kv for w in x + y + A + B):
            d, p = weat(x, y, A, B, kv)
            filas.append({"embeddings": nombre_emb, "prueba": prueba, "tamaño del efecto d": d, "valor p": p})
pd.DataFrame(filas).round(3)

,embeddings,prueba,tamaño del efecto d,valor p
0,GloVe,carrera vs. familia,1.665,0.000
1,GloVe,control: instrumentos vs. herramientas,0.286,0.291
2,Word2Vec text8,carrera vs. familia,1.734,0.000


**Interpretación:**

- En la prueba carrera/familia, el tamaño del efecto es **grande y positivo** (en psicología, $d > 0.8$ se considera un efecto grande) con un valor $p$ muy pequeño: las palabras de carrera se asocian más con nombres masculinos y las de familia con nombres femeninos. Los embeddings reproducen el estereotipo presente en los textos con que fueron entrenados, tal como encontraron Caliskan et al.
- En GloVe, la prueba de control da un efecto pequeño y no significativo ($p \approx 0.3$), como se esperaba. En el Word2Vec de text8 la prueba de control no aparece porque alguna de sus palabras (herramientas poco frecuentes) no está en el vocabulario: un recordatorio de que las listas de WEAT deben verificarse para cada modelo. Con 8 palabras por grupo el test tiene poca potencia, y la elección de las listas influye mucho en el resultado.
- Limitaciones de WEAT: depende de listas de palabras elegidas a mano (en inglés y con nombres de una cultura concreta), y un sesgo medido en los embeddings no implica automáticamente un sesgo en el sistema final, ni su ausencia lo descarta. Aun así, es una herramienta útil para auditar representaciones antes de usarlas.

---

<a id="ejercicio-4"></a>
<h2 style="color: #007ACC;">Ejercicio 4. Seq2Seq a nivel de caracteres: normalizar fechas</h2>

**Objetivo:** construir y entrenar un codificador-decodificador por caracteres que convierta fechas escritas en español al formato ISO 8601.

Usaremos datos **sintéticos**, generados con el código dado. Cada ejemplo es una fecha escrita en uno de 8 formatos y su versión `AAAA-MM-DD`:

```text
23 de marzo de 2021        → 2021-03-23
martes 23 de marzo de 2021 → 2021-03-23
23/03/2021                 → 2021-03-23
marzo 23, 2021             → 2021-03-23
```

1. Ejecuta la celda de generación de datos y revisa ejemplos de cada formato.
2. Construye los vocabularios de caracteres de entrada y de salida (la salida incluye los tokens de inicio `^` y fin `$`) y convierte los textos en matrices de índices con relleno 0.
3. Construye el modelo con la API funcional: codificador `Embedding(…, 32)` + `Bidirectional(GRU(128))` con `return_state=True` (concatena los dos estados finales, 256 dimensiones); decodificador `Embedding(…, 32)` + `GRU(256, return_sequences=True)` con ese estado inicial + `Dense` sobre el vocabulario de salida. Entrénalo con *teacher forcing*.
4. Implementa la decodificación voraz **por lotes** (todas las fechas de prueba a la vez; la salida tiene siempre 10 caracteres más el fin).
5. Reporta la **exactitud exacta** (la fecha completa correcta) en prueba, global y por formato.

In [8]:
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
ABREV = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

FORMATOS = {
    "largo": lambda f: f"{f.day} de {MESES[f.month - 1]} de {f.year}",
    "barras": lambda f: f"{f.day:02d}/{f.month:02d}/{f.year}",
    "guiones_año_corto": lambda f: f"{f.day:02d}-{f.month:02d}-{f.year % 100:02d}",
    "mes_primero": lambda f: f"{MESES[f.month - 1]} {f.day}, {f.year}",
    "con_día_semana": lambda f: f"{DIAS[f.weekday()]} {f.day} de {MESES[f.month - 1]} de {f.year}",
    "abreviado": lambda f: f"{f.day} {ABREV[f.month - 1]} {f.year}",
    "con_artículo": lambda f: f"el {f.day} de {MESES[f.month - 1]} del {f.year}",
    "puntos": lambda f: f"{f.day:02d}.{f.month:02d}.{f.year}",
}


def generar_fechas(n, formatos, rng):
    """Fechas al azar entre 1950 y 2049 (así el año de dos cifras no es ambiguo)."""
    inicio = datetime.date(1950, 1, 1).toordinal()
    fin = datetime.date(2049, 12, 31).toordinal()
    datos = []
    for _ in range(n):
        f = datetime.date.fromordinal(int(rng.integers(inicio, fin + 1)))
        nombre = formatos[int(rng.integers(len(formatos)))]
        texto = FORMATOS[nombre](f)
        if rng.random() < 0.3:
            texto = texto.capitalize()
        datos.append((texto, f.isoformat(), nombre))
    return datos


rng_datos = np.random.default_rng(SEED)
todos = list(FORMATOS)
train_f = generar_fechas(40_000, todos, rng_datos)
val_f = generar_fechas(2_000, todos, rng_datos)
test_f = generar_fechas(2_000, todos, rng_datos)
for texto, iso, nombre in train_f[:8]:
    print(f"{nombre:<18} {texto:<32} → {iso}")

con_artículo       el 4 de diciembre del 1958       → 1958-12-04
con_artículo       el 20 de abril del 1993          → 1993-04-20
largo              23 de febrero de 1970            → 1970-02-23
con_artículo       el 30 de julio del 2023          → 2023-07-30
barras             28/04/2001                       → 2001-04-28
guiones_año_corto  14-01-00                         → 2000-01-14
abreviado          27 feb 2028                      → 2028-02-27
mes_primero        Julio 17, 2004                   → 2004-07-17


In [9]:
chars_in = ["<pad>"] + sorted({c for t, _, _ in train_f for c in t} | set("áéíóúABCDEFGHIJKLMNOPQRSTUVWXYZ"))
chars_out = ["<pad>", "^", "$"] + sorted(set("0123456789-"))
ci, co = {c: i for i, c in enumerate(chars_in)}, {c: i for i, c in enumerate(chars_out)}
LONG_IN, LONG_OUT = 40, 12  # salida: ^ + 10 caracteres + $


def codificar(datos):
    X = np.zeros((len(datos), LONG_IN), dtype="int64")
    Y = np.zeros((len(datos), LONG_OUT), dtype="int64")
    for k, (texto, iso, _) in enumerate(datos):
        ids = [ci.get(c, 0) for c in texto[:LONG_IN]]
        X[k, :len(ids)] = ids
        Y[k] = [co[c] for c in "^" + iso + "$"]
    return X, Y[:, :-1], Y[:, 1:]   # entrada del codificador, entrada del decodificador, salida


Xe_tr, Xd_tr, Y_tr = codificar(train_f)
Xe_va, Xd_va, Y_va = codificar(val_f)
Xe_te, Xd_te, Y_te = codificar(test_f)
print(f"Vocabulario de entrada: {len(chars_in)} | de salida: {len(chars_out)}")

# Codificador
ent_cod = keras.Input(shape=(None,), dtype="int64")
x = layers.Embedding(len(chars_in), 32, mask_zero=True)(ent_cod)
_, h_ida, h_vuelta = layers.Bidirectional(layers.GRU(128, return_state=True))(x)
estado = layers.Concatenate()([h_ida, h_vuelta])

# Decodificador
ent_dec = keras.Input(shape=(None,), dtype="int64")
emb_dec = layers.Embedding(len(chars_out), 32)
gru_dec = layers.GRU(256, return_sequences=True, return_state=True)
densa = layers.Dense(len(chars_out))
salida, _ = gru_dec(emb_dec(ent_dec), initial_state=estado)
modelo_fechas = keras.Model([ent_cod, ent_dec], densa(salida))
modelo_fechas.compile(optimizer="adam",
                      loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])

inicio = time.perf_counter()
modelo_fechas.fit([Xe_tr, Xd_tr], Y_tr, validation_data=([Xe_va, Xd_va], Y_va), epochs=10, batch_size=128, verbose=2)
print(f"Entrenado en {time.perf_counter() - inicio:.0f} s")

Vocabulario de entrada: 68 | de salida: 14


Epoch 1/10


313/313 - 9s - 28ms/step - accuracy: 0.6841 - loss: 0.8728 - val_accuracy: 0.8055 - val_loss: 0.4879


Epoch 2/10


313/313 - 7s - 23ms/step - accuracy: 0.8307 - loss: 0.4297 - val_accuracy: 0.8529 - val_loss: 0.3548


...

Epoch 9/10


313/313 - 7s - 23ms/step - accuracy: 1.0000 - loss: 0.0010 - val_accuracy: 1.0000 - val_loss: 8.7609e-04


Epoch 10/10


313/313 - 7s - 23ms/step - accuracy: 1.0000 - loss: 7.0145e-04 - val_accuracy: 1.0000 - val_loss: 6.3183e-04


Entrenado en 74 s


In [10]:
codificador_f = keras.Model(ent_cod, estado)
prev = keras.Input(shape=(1,), dtype="int64")
h_prev = keras.Input(shape=(256,))
out, h_nuevo = gru_dec(emb_dec(prev), initial_state=h_prev)
paso_f = keras.Model([prev, h_prev], [densa(out), h_nuevo])


def decodificar_lote(Xe):
    h = codificador_f.predict(Xe, verbose=0)
    actual = np.full((len(Xe), 1), co["^"])
    salida = []
    for _ in range(LONG_OUT - 1):
        logits, h = paso_f.predict([actual, h], verbose=0)
        actual = logits[:, -1].argmax(axis=1)[:, None]
        salida.append(actual[:, 0])
    salida = np.stack(salida, axis=1)
    return ["".join(chars_out[i] for i in fila).split("$")[0] for fila in salida]


pred_te = decodificar_lote(Xe_te)
resultado = pd.DataFrame({"formato": [n for _, _, n in test_f], "correcta": [p == iso for p, (_, iso, _) in zip(pred_te, test_f)]})
print(f"Exactitud exacta en prueba: {resultado.correcta.mean():.4f}")
resultado.groupby("formato").correcta.mean().round(4).to_frame("exactitud")

Exactitud exacta en prueba: 1.0000


,exactitud
formato,
abreviado,1.0
barras,1.0
con_artículo,1.0
con_día_semana,1.0
guiones_año_corto,1.0
largo,1.0
mes_primero,1.0
puntos,1.0


**Interpretación:** la tarea parece trivial para una persona, pero el modelo no conoce ninguna regla: tiene que **aprender** que *marzo* corresponde a `03`, que en *"23/03/2021"* el día va primero, que en *"23-03-21"* el año `21` es `2021` y no `1921`, y que el día de la semana es irrelevante. Con 40 000 ejemplos y poco más de un minuto de entrenamiento, el Seq2Seq por caracteres convierte correctamente **todas** las fechas de prueba, en los 8 formatos.

---

<a id="ejercicio-5"></a>
<h2 style="color: #007ACC;">Ejercicio 5. Generalización a formatos no vistos</h2>

**Objetivo:** evaluar si el Seq2Seq aprendió reglas generales o solo los formatos que vio.

1. Genera un nuevo conjunto de entrenamiento con todos los formatos **excepto** `"mes_primero"` (*"marzo 23, 2021"*) y `"puntos"` (*"23.03.2021"*). Entrena un modelo nuevo con la misma arquitectura (puedes envolver la construcción del ejercicio 4 en una función).
2. Evalúa la exactitud exacta por formato sobre `test_f`, que sí contiene los dos formatos excluidos.
3. Para los formatos no vistos, muestra 8 predicciones erróneas. ¿Qué parte de la fecha falla? ¿Por qué crees que el modelo generaliza mejor a uno de los dos formatos?
4. ¿Qué implica esto para el uso de modelos Seq2Seq (y, más adelante, de LLM) en tareas de normalización de datos reales?

In [11]:
def construir_y_entrenar(Xe, Xd, Y, epocas=10):
    ent_c = keras.Input(shape=(None,), dtype="int64")
    x = layers.Embedding(len(chars_in), 32, mask_zero=True)(ent_c)
    _, h1, h2 = layers.Bidirectional(layers.GRU(128, return_state=True))(x)
    est = layers.Concatenate()([h1, h2])
    ent_d = keras.Input(shape=(None,), dtype="int64")
    e_dec, g_dec, d_out = layers.Embedding(len(chars_out), 32), layers.GRU(256, return_sequences=True, return_state=True), layers.Dense(len(chars_out))
    s, _ = g_dec(e_dec(ent_d), initial_state=est)
    m = keras.Model([ent_c, ent_d], d_out(s))
    m.compile(optimizer="adam", loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))
    m.fit([Xe, Xd], Y, epochs=epocas, batch_size=128, verbose=0)
    # Modelos de inferencia
    cod = keras.Model(ent_c, est)
    p_in, h_in = keras.Input(shape=(1,), dtype="int64"), keras.Input(shape=(256,))
    o, h_out = g_dec(e_dec(p_in), initial_state=h_in)
    return cod, keras.Model([p_in, h_in], [d_out(o), h_out])


def decodificar_con(cod, paso, Xe):
    h = cod.predict(Xe, verbose=0)
    actual, salida = np.full((len(Xe), 1), co["^"]), []
    for _ in range(LONG_OUT - 1):
        logits, h = paso.predict([actual, h], verbose=0)
        actual = logits[:, -1].argmax(axis=1)[:, None]
        salida.append(actual[:, 0])
    return ["".join(chars_out[i] for i in fila).split("$")[0] for fila in np.stack(salida, axis=1)]


vistos = [f for f in FORMATOS if f not in ("mes_primero", "puntos")]
train_parcial = generar_fechas(40_000, vistos, np.random.default_rng(SEED + 1))
Xe_p, Xd_p, Y_p = codificar(train_parcial)
keras.utils.set_random_seed(SEED)
cod_p, paso_p = construir_y_entrenar(Xe_p, Xd_p, Y_p)

pred_p = decodificar_con(cod_p, paso_p, Xe_te)
res_p = pd.DataFrame({"formato": [n for _, _, n in test_f],
                      "visto en entrenamiento": [n in vistos for _, _, n in test_f],
                      "correcta": [p == iso for p, (_, iso, _) in zip(pred_p, test_f)]})
res_p.groupby(["visto en entrenamiento", "formato"]).correcta.mean().round(4).to_frame("exactitud")

exactitud
visto en entrenamiento formato                     
False                  mes_primero           0.0287
                       puntos                0.3915
True                   abreviado             1.0000
                       barras                1.0000
                       con_artículo          1.0000
                       con_día_semana        1.0000
                       guiones_año_corto     1.0000
                       largo                 1.0000

In [12]:
errores = [(t, iso, p, n) for p, (t, iso, n) in zip(pred_p, test_f) if n not in vistos and p != iso]
pd.DataFrame(errores[:8], columns=["entrada", "correcta", "predicción", "formato"])

,entrada,correcta,predicción,formato
0,12.09.1960,1960-09-12,1960-12-12,puntos
1,"abril 27, 1956",1956-04-27,1956-07-23,mes_primero
2,"Agosto 27, 1989",1989-08-27,1989-08-23,mes_primero
3,08.06.1952,1952-06-08,1952-09-08,puntos
4,27.03.1952,1952-03-27,1952-05-27,puntos
5,"noviembre 29, 2034",2034-11-29,2034-09-26,mes_primero
6,"mayo 10, 2028",2028-05-10,2028-12-13,mes_primero
7,16.09.2044,2044-09-16,2044-12-16,puntos


**Interpretación:**

- En los formatos **vistos** la exactitud sigue siendo muy alta. En los **no vistos** cae, y de forma distinta según el formato.
- *"23.03.2021"* se parece mucho a *"23/03/2021"* (ya visto): el modelo solo tiene que tratar el punto como otra barra, así que suele generalizar razonablemente.
- *"marzo 23, 2021"* invierte el orden habitual de día y mes. El modelo aprendió que el número que aparece **antes** del mes es el día, y ahora el número aparece **después**. Los errores típicos están en el día (y a veces en el año, que también cambia de posición relativa).
- **Implicación:** un modelo neuronal aprende las regularidades de **sus datos de entrenamiento**, no las reglas que a nosotros nos parecen obvias. En aplicaciones reales hay que evaluar con datos que reflejen la variedad de lo que llegará en producción (distribución fuera de la de entrenamiento), y para tareas con reglas claras y verificables (como fechas) muchas veces un sistema de reglas, o un modelo combinado con validación por reglas, es más confiable. Los LLM generalizan mucho mejor gracias a su preentrenamiento, pero el principio de evaluar con datos representativos sigue vigente.

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador). [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/)
- Zhang, A., Lipton, Z. C., Li, M. & Smola, A. J. *Dive into Deep Learning*. [d2l.ai](https://d2l.ai/)
- Levy, O. & Goldberg, Y. (2014). Linguistic Regularities in Sparse and Explicit Word Representations. *CoNLL*.
- Caliskan, A., Bryson, J. J. & Narayanan, A. (2017). Semantics derived automatically from language corpora contain human-like biases. *Science*, 356(6334).
- Mikolov, T. et al. (2013). Efficient Estimation of Word Representations in Vector Space. *ICLR Workshop*.